# Session 6 — Demo 3 — Every metric on one page

**What this demo covers.** Sessions 5 and 6 graded models in many ways: the average miss, R², accuracy, recall, precision, AUC, the cost of a threshold. This page puts them side by side: how each one is computed, what question it answers, when to use it and when it lies. It starts from the number the business really cares about, because every model metric is a stand-in for it, and it shows how to write your own metric when no standard one fits. Toy tables where every answer can be checked by eye, then two reference tables and five questions for choosing a metric.

**Data.** None: toy tables only.

**How to run.** Locally or in Colab, top to bottom. Nothing to download.

## 0. Setup

In [1]:
import pandas as pd

## 1. Three numbers: the business goal, the model metric, the loss

**The business goal comes first.** No business asks for a high AUC. It asks for money not lost to defaults, rooms not left empty, analyst hours saved, clients kept. That is what the model is for. These **business metrics** are often hard to compute on a test set: they depend on prices, on what people do with a flag, on the months after the decision.

**The model metric is a stand-in.** So we pick a number we *can* compute on the test part from the truth and the model's output, and that moves together with the business goal. Recall, when a missed default is what hurts. Expected cost, when we can name the prices. A good stand-in rises when the business number improves. A bad one can rise while the business loses money: accuracy did exactly that with 3% bankrupt firms. Choosing the stand-in is a decision you make with the business, not a default you take from a library.

**The loss is for the computer.** A model learns by making one number as small as possible on the training data. That number is the **loss**. Session 5's least-squares line made the sum of squared errors as small as possible. Logistic regression and CatBoost minimise **log-loss**, a penalty that grows fast when the model is confidently wrong. The loss is chosen so that the computer can learn quickly and reliably. In Demo 1, CatBoost was trained on log-loss, and we judged it by the cost of a threshold.

**Choose in this order: the business goal, then the model metric that brings you closest to it, and let the library choose the loss.**

## 2. Regression: how far off are the numbers?

🧪 **Toy example:** five warehouse tasks, the real time each took and a model's prediction, in minutes. One task is a big miss, say a picker who got stuck.

In [2]:
toy_tasks = pd.DataFrame({
    "actual_minutes": [10, 12, 8, 15, 30],
    "predicted_minutes": [11, 12, 10, 14, 18],
})
toy_tasks["error"] = toy_tasks["actual_minutes"] - toy_tasks["predicted_minutes"]
toy_tasks

,actual_minutes,predicted_minutes,error
0,10,11,-1
1,12,12,0
2,8,10,-2
3,15,14,1
4,30,18,12


The errors are −1, 0, −2, 1 and 12. Four ways to turn them into one number, by hand first:

- **MAE**, mean absolute error: drop the signs and average. (1 + 0 + 2 + 1 + 12) / 5 = **3.2 minutes**. Session 5's "average miss" was exactly this.
- **RMSE**, root mean squared error: square the errors, average, take the square root. √((1 + 0 + 4 + 1 + 144) / 5) = √30 = **5.5 minutes**. Squaring lets the big miss dominate: it brings 144 of the 150.
- **MAPE**, mean absolute percentage error: each miss as a share of the real value, then the average. (0.10 + 0 + 0.25 + 0.07 + 0.40) / 5 = **16%**.
- **R²**, the share of the spread explained (session 5). The mean of the real values is 15, and the squared distances from it add up to 25 + 9 + 49 + 0 + 225 = 308. The model's squared errors add up to 150. R² = 1 − 150 / 308 = **0.51**. R² of 1 is perfect, 0 is no better than always predicting the mean, and below 0 is worse than the mean.

scikit-learn has a function for each: `mean_absolute_error`, `root_mean_squared_error`, `mean_absolute_percentage_error` (it returns a share, so 0.16 means 16%) and `r2_score`. Each takes the real values first, then the predictions.

In [3]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)

actual_minutes = toy_tasks["actual_minutes"]
predicted_minutes = toy_tasks["predicted_minutes"]
print(
    f"MAE:  {mean_absolute_error(actual_minutes, predicted_minutes):.1f} minutes\n"
    f"RMSE: {root_mean_squared_error(actual_minutes, predicted_minutes):.1f} minutes\n"
    f"MAPE: {mean_absolute_percentage_error(actual_minutes, predicted_minutes):.0%}\n"
    f"R²:   {r2_score(actual_minutes, predicted_minutes):.2f}"
)

MAE:  3.2 minutes
RMSE: 5.5 minutes
MAPE: 16%
R²:   0.51


Same numbers as by hand. Notice how differently they react to the one stuck picker: MAE says "a typical miss of 3 minutes", RMSE says 5.5, because it counts that one miss many times over.

## 3. Classification: the six toy firms once more

🧪 Session 5's six made-up firms: two bankrupt, three flagged. By eye: caught 1, missed 1, false alarms 2, correctly cleared 2.

- **accuracy** = right calls / all calls = (1 + 2) / 6 = 0.50;
- **recall** = caught / all bankrupt = 1 / 2 = 0.50;
- **precision** = caught / all flags = 1 / 3 = 0.33;
- **F1** puts precision and recall into one number that is high only when *both* are high: 2 × P × R / (P + R) = 2 × 0.33 × 0.5 / 0.83 = **0.40**. It sits closer to the lower of the two, on purpose.

The scikit-learn functions are `accuracy_score`, `recall_score`, `precision_score` and `f1_score`: truth first, then the calls.

In [4]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

toy_flags = pd.DataFrame({
    "actually_bankrupt": [1, 1, 0, 0, 0, 0],
    "called_bankrupt": [1, 0, 1, 1, 0, 0],
})
actually_bankrupt = toy_flags["actually_bankrupt"]
called_bankrupt = toy_flags["called_bankrupt"]
print(
    f"accuracy:  {accuracy_score(actually_bankrupt, called_bankrupt):.2f}\n"
    f"recall:    {recall_score(actually_bankrupt, called_bankrupt):.2f}\n"
    f"precision: {precision_score(actually_bankrupt, called_bankrupt):.2f}\n"
    f"F1:        {f1_score(actually_bankrupt, called_bankrupt):.2f}"
)

accuracy:  0.50
recall:    0.50
precision: 0.33
F1:        0.40


## 4. When no standard metric fits, write your own

Every metric on this page is somebody's answer to somebody's business question. Your question can be different. Say a credit team can review only a few firms a week, the ones with the highest scores. What the team cares about is: **of all the bankrupt firms, what share lands in our review queue?** Recall at 0.5 doesn't answer that, and neither does AUC.

A metric is just a function: the truth and the model's output in, one number out. So write it. The function below sorts the firms by score, keeps the top `queue_size` of them, and divides the bankrupt firms in the queue by all bankrupt firms.

🧪 **Toy example:** Demo 1's five firms, bankrupt at 0.8 and 0.3, healthy at 0.6, 0.2 and 0.1. By eye: a queue of 1 holds the 0.8, so it catches 1 of 2 bankrupt firms, 0.5. A queue of 2 adds the healthy 0.6: still 0.5. A queue of 3 adds the 0.3: 1.0.

In [5]:
def compute_share_caught_in_queue(y_true, scores, queue_size):
    """Share of all bankrupt firms among the queue_size highest scores."""
    firms = pd.DataFrame({
        "actually_bankrupt": y_true,
        "score": scores,
    })
    review_queue = firms.sort_values("score", ascending=False).head(queue_size)
    return review_queue["actually_bankrupt"].sum() / firms["actually_bankrupt"].sum()


toy_scores = pd.DataFrame({
    "actually_bankrupt": [1, 1, 0, 0, 0],
    "score": [0.8, 0.3, 0.6, 0.2, 0.1],
})
for queue_size in [1, 2, 3]:
    share_caught = compute_share_caught_in_queue(
        toy_scores["actually_bankrupt"],
        toy_scores["score"],
        queue_size,
    )
    print(f"review queue of {queue_size}: {share_caught:.0%} of bankrupt firms caught")

review queue of 1: 50% of bankrupt firms caught
review queue of 2: 50% of bankrupt firms caught
review queue of 3: 100% of bankrupt firms caught


Same as by eye. This metric has a name, **recall at k**, and versions of it are common in credit and fraud teams, where the review capacity is fixed. Demo 1's expected cost was a custom metric too: the business named the prices, and we wrote them into a function.

Three rules for a metric of your own:

- **It must be computable on the test part** from the truth and the model's output alone.
- **Someone outside the data team must agree that it moves with the money.** If the head of credit wouldn't care about it, it is the wrong metric.
- **Check it on a toy where you know the answer**, exactly as above, before you trust it on real data.

To use your own metric inside cross-validation, scikit-learn's `make_scorer` wraps the function so that `cross_val_score(..., scoring=...)` accepts it. Ask an assistant for that boilerplate, then check the result against your toy.

## 5. The map: classification

The classification metrics of sessions 5 and 6 in one table. TP, FN, FP and TN are the four cells of the confusion matrix (Demo 1, section 3).

| metric | the question it answers | computed as | use it when | it misleads when |
|---|---|---|---|---|
| accuracy | how many calls were right? | (TP + TN) / all | classes are balanced and both mistakes cost the same | the positive class is rare: "everyone healthy" scores 97% |
| recall, or true positive rate | of the real positives, how many did we catch? | TP / (TP + FN) | a miss is dear: screening, fraud, defaults | read alone: flag everyone and it is 100% |
| precision | of our flags, how many were real? | TP / (TP + FP) | a false alarm is dear: a spam filter, costly manual checks | read alone: flag one sure case and it is 100% |
| F1 | are precision and recall both decent? | 2 × P × R / (P + R) | you need one number and can't name the prices | you can name the prices: F1 treats both mistakes as equally bad |
| false positive rate | of the real negatives, how many did we flag? | FP / (FP + TN) | the x-axis of the ROC curve | negatives are many: 5% of 1,980 healthy firms is 99 false alarms |
| ROC-AUC | how well does the model order the cases, over all thresholds? | area under the ROC curve; the chance that a random positive outranks a random negative | comparing models before a threshold is chosen | you need a decision: it doesn't say where to cut, and with rare positives it can look great while precision is poor |
| PR-AUC, or average precision | the same idea, on the precision–recall curve | area under precision plotted against recall | positives are rare and false alarms matter | harder to read: its "coin" level is the share of positives, not 0.5 |
| expected cost | what does this threshold cost the business? | missed × ratio + false alarms | you can name the price of each mistake | the ratio is a guess: show how the threshold moves when it changes |
| recall at k | what share of the positives lands in a queue of fixed size? | positives among the k highest scores / all positives | the team can review only k cases | the queue size changes: the number changes with it |
| calibration by score bands | can the score be read as a probability? | average score against the real share, band by band | the number itself feeds a formula: expected loss, a price, a reserve | the bands hold few cases: the shares are noisy |

## 6. The map: regression

| metric | the question it answers | computed as | use it when | it misleads when |
|---|---|---|---|---|
| MAE | how big is a typical miss, in units? | the average of the errors without their signs | by default: it reads aloud, "4.9 minutes" | one huge miss matters far more than many small ones |
| RMSE | how big is a typical miss, with big misses weighted up? | the square root of the average squared error | a big miss is much worse than two small ones | a few outliers dominate everything |
| MAPE | how big is a typical miss, in percent? | the average of each miss divided by the real value | the items differ a lot in size | real values near zero: the percent explodes |
| R² | what share of the spread does the model explain? | 1 − (squared errors) / (squared distances from the mean) | how much better than always predicting the mean | comparing different datasets; a high R² can still hide big misses in units |

**This is not the full list.** These are the metrics you will meet most often, and there are many more: every field adds its own, and teams invent new ones for their decisions. A few you may hear in finance:

- **Gini** = 2 × AUC − 1: the same ordering grade on a scale from 0 (a coin) to 1 (perfect). Credit-scoring reports often quote it instead of AUC. Demo 1's boosting: 2 × 0.949 − 1 = 0.90.
- **KS statistic:** the largest gap, over all thresholds, between the share of bankrupt firms flagged and the share of healthy firms flagged. On the ROC chart it is the largest vertical distance between the curve and the diagonal.
- **Log-loss** and the **Brier score** grade the probabilities themselves. The Brier score is the average squared difference between the score and the 0/1 truth.
- **SMAPE**, **WAPE** and quantile loss, among others, for regression.

When you meet a metric you don't know, ask it three questions: what does it answer, what does it ignore, and does it move with the business number?

## 7. Which metric? Five questions

1. **What business number should the model move, and through which decision?** Money lost to defaults, through a flag for review; staff hours, through tomorrow's shift plan.
2. **What does each mistake cost?** If you can name prices, use the expected cost and choose the threshold with it. If one mistake clearly dominates, watch recall or precision, and always report the other one next to it.
3. **Is the number itself used?** Then check calibration before it goes into a formula.
4. **Are you comparing models before choosing a threshold?** Use AUC, or PR-AUC when positives are rare; then pick the threshold by cost.
5. **Does no standard metric match the decision?** Write your own (section 4), check it on a toy, and agree it with the business.

And always put a **baseline** next to your metric: "everyone healthy" for classification, "always predict the mean" for regression, which is R² = 0.

**Further reading, in Russian:** [Yandex ML handbook, "Classification and regression metrics"](https://contest.yandex.ru/tracks/ml/model-evaluation/classification-and-regression-metrics) covers every metric on this page and many more; [Yandex ML handbook, "How to evaluate probabilities"](https://contest.yandex.ru/tracks/ml/probabilistic-models/probability-estimation) covers calibration.

## Try it yourself

1. Change the stuck picker's real time from 30 to 60 minutes. Which regression metric jumps the most, and why?
2. In the six toy firms, flag every firm. Compute accuracy, recall, precision and F1 by eye first, then check with the code.
3. Pick one HW2 dataset. Which row of the classification map would its business owner care about most, and which would mislead them?
4. Invent a metric for that HW2 business: what would the decision maker count at the end of a month? Write it as a function and check it on a five-row toy.

In [6]:
# your turn

## Recap

- **Start from the business goal.** A model metric is a stand-in for it: pick the one that moves with the money. The **loss** is what the model minimises while it learns; let the library choose it.
- Regression: **MAE** is the typical miss in units (session 5's average miss), **RMSE** weighs big misses up, **MAPE** is in percent and breaks near zero, **R²** compares the model with always predicting the mean.
- Classification: **accuracy** misleads when one class is rare; **recall** and **precision** each answer one question and are easy to game alone; **F1** joins them without prices; **ROC-AUC** and **PR-AUC** grade the order over all thresholds; **expected cost** chooses the threshold; **calibration** checks whether a score is a probability.
- The maps are not the full list. When no metric fits your decision, **write your own**, check it on a toy, and agree it with the business.
- Always report a baseline next to the metric.